In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, recall_score, precision_score
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

# Load and prepare data
data = pd.read_csv('Klasifikasi Katarak.csv', header=None, decimal=',')
data.columns = ['Feature1', 'Feature2', 'Feature3', 'Label']

# Prepare features and labels
X = data[['Feature1', 'Feature2', 'Feature3']].values
y = data['Label'].values

# Encode labels
le = LabelEncoder()
y = le.fit_transform(y)

# Scale features
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Simplified parameter grid
param_distributions = {
    'hidden_layer_sizes': [(200, 100), (175, 85), (160, 90), (300, 150), (250, 125), (200, 150), (300, 200), (250, 200), (200, 200)],
    # Remove relu since tanh/logistic work better with lbfgs
    'activation': ['tanh', 'logistic'],
    'solver': ['lbfgs'],
    'batch_size': [16, 32, 64, 128, 256, 512],
    # Added more learning rates
    'learning_rate_init': [0.1, 0.01, 0.001, 0.0001, 0.00001],
    'early_stopping': [True],  # Unchanged
    'validation_fraction': [0.1, 0.15, 0.2, 0.25, 0.3],
    'momentum': [0.7, 0.8, 0.85, 0.9, 0.95, 0.99],
}

# Create base model with increased max_iter
base_model = MLPClassifier(max_iter=3000, random_state=42, tol=1e-6)

# Use RandomizedSearchCV instead of GridSearchCV
random_search = RandomizedSearchCV(
    base_model, param_distributions, n_iter=50,
    cv=StratifiedKFold(n_splits=5), scoring='accuracy', n_jobs=-1, random_state=42)
random_search.fit(X, y)

# Print best parameters
print("Best Parameters:", random_search.best_params_)
print(f"Best Cross-Validation Score: {random_search.best_score_ * 100:.2f}%")

# Split data with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Train final model with best parameters
best_model = MLPClassifier(
    **random_search.best_params_, max_iter=1000, random_state=42)
best_model.fit(X_train, y_train)

# Predictions
y_pred = best_model.predict(X_test)

# Calculate and print accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"\nTest Set Accuracy: {accuracy * 100:.2f}%")

# Calculate sensitivity and specificity for each class
recall = recall_score(y_test, y_pred, average=None)
specificity = []
cm = confusion_matrix(y_test, y_pred)
for i in range(len(le.classes_)):
    tn = cm.sum() - (cm[i, :].sum() + cm[:, i].sum() - cm[i, i])
    fp = cm[:, i].sum() - cm[i, i]
    specificity.append(tn / (tn + fp))
for idx, label in enumerate(le.classes_):
    print(f"{label} - Sensitivity: {recall[idx] *
          100:.2f}%, Specificity: {specificity[idx] * 100:.2f}%")

# Print detailed classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

# Visualize confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_,
            yticklabels=le.classes_)
plt.title(f'Confusion Matrix (Accuracy: {accuracy * 100:.2f}%)')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

# Perform stratified k-fold cross validation
cv_scores = cross_val_score(best_model, X, y, cv=StratifiedKFold(n_splits=5))
print(f"\nCross-validation scores: {cv_scores}")
print(f"Average CV Score: {cv_scores.mean() *
      100:.2f}% (+/- {cv_scores.std() * 2 * 100:.2f}%)")